In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

In [2]:
# Initialize Spark
builder = SparkSession.builder \
    .appName("Verify-Bronze") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
print("=" * 80)
print("BRONZE LAYER VERIFICATION")
print("=" * 80)

# Read all tables
customers = spark.read.format("delta").load("s3a://bronze/customers")
accounts = spark.read.format("delta").load("s3a://bronze/accounts")
transactions = spark.read.format("delta").load("s3a://bronze/transactions")
aml_alerts = spark.read.format("delta").load("s3a://bronze/aml_alerts")

print(f"\n✅ Customers: {customers.count():,} records")
print(f"✅ Accounts: {accounts.count():,} records")
print(f"✅ Transactions: {transactions.count():,} records")
print(f"✅ AML Alerts: {aml_alerts.count():,} records")

BRONZE LAYER VERIFICATION

✅ Customers: 10,000 records
✅ Accounts: 28,299 records
✅ Transactions: 100,000 records
✅ AML Alerts: 5,000 records


In [5]:
# Verify audit columns exist
print("\n📋 Audit Columns Present:")
audit_cols = ['_ingestion_timestamp', '_source_file', '_source_system', '_data_classification', '_data_residency', '_record_hash']
for col in audit_cols:
    print(f"  ✓ {col}")


📋 Audit Columns Present:
  ✓ _ingestion_timestamp
  ✓ _source_file
  ✓ _source_system
  ✓ _data_classification
  ✓ _data_residency
  ✓ _record_hash


In [6]:
# Sample query: High-value transactions from last 30 days
print("\n🔍 Sample Query: High-Value Transactions (Last 30 Days)")
from pyspark.sql.functions import col, current_date, datediff

high_value_txns = transactions.filter(
    (col("amount") > 50000) & 
    (datediff(current_date(), col("transaction_date")) <= 30)
)

print(f"Found {high_value_txns.count():,} high-value transactions")
high_value_txns.select("transaction_id", "amount", "currency", "country", "transaction_date").show(5)


🔍 Sample Query: High-Value Transactions (Last 30 Days)
Found 3,330 high-value transactions
+---------------+---------+--------+-------+----------------+
| transaction_id|   amount|currency|country|transaction_date|
+---------------+---------+--------+-------+----------------+
|TXN5FECFF81C58B| 98156.35|     EUR|    UAE|      2026-02-16|
|TXNA4AC4D59D92B| 80270.41|     USD|    UAE|      2026-02-16|
|TXN9EA6AAC944F7|246178.77|     USD|    UAE|      2026-02-16|
|TXN78C0B7203618| 56726.86|     EUR|    UAE|      2026-02-16|
|TXNFD9A12D72F45|  59350.5|     USD|    UAE|      2026-02-16|
+---------------+---------+--------+-------+----------------+
only showing top 5 rows



In [7]:
# Demonstrate time travel
print("\n⏰ Delta Lake Time Travel:")
from delta.tables import DeltaTable

customers_table = DeltaTable.forPath(spark, "s3a://bronze/customers")
history = customers_table.history()
print(f"Total versions: {history.count()}")
history.select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)


⏰ Delta Lake Time Travel:
Total versions: 1
+-------+-------------------+---------+------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                  |
+-------+-------------------+---------+------------------------------------------------------------------+
|0      |2026-02-27 21:02:48|WRITE    |{numFiles -> 1, numOutputRows -> 10000, numOutputBytes -> 1113580}|
+-------+-------------------+---------+------------------------------------------------------------------+



In [8]:
print("\n" + "=" * 80)
print("✅ BRONZE LAYER VERIFICATION COMPLETE!")
print("=" * 80)

spark.stop()


✅ BRONZE LAYER VERIFICATION COMPLETE!
